
# CS0073 Technical Assessment, Module 2: PRACTICE VERSION
## Diagnostic Analytics and Statistical Inference

This is the practice copy of the full notebook. The code has been removed. For each step:

1. Read the **Your Turn** instructions and the **Hint**.
2. Write the code in the empty cell below it, using the exact variable names given.
3. Run the **Check** cell under it. `[OK]` means your number matches the full version.
   `[X]` means it does not match yet; re-read the hint and try again.
4. Say the **Defend It** prompt out loud in one or two sentences. The live reasoning check is done
   without AI, so this is the part that counts.

Keep `Customer_Satisfaction.xlsx` in the same folder as this notebook. Run the cells from top to bottom.


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import os

pd.set_option("display.precision", 4)
ALPHA = 0.05  # significance level used for every test in this notebook

# Chart style: one quiet look for every figure
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": False,
    "font.size": 10,
})
FIG_DIR = "PAMESA - M2-Technical - Figures"
os.makedirs(FIG_DIR, exist_ok=True)

The next cell loads the answer key used by the check cells. Run it, but do not read the numbers before you try each step.

In [ ]:
EXPECTED = {"raw_shape": [405.0, 17.0], "n_rows": 405.0, "n_missing": 0.0, "n_duplicate_ids": 0.0, "n_out_of_range": 0.0, "desc_means": [3.639506172839506, 4.637037037037037, 4.508641975308642, 5.762962962962963, 4.338271604938272, 4.696296296296296, 5.311111111111111, 3.5530864197530865, 4.523456790123457, 5.209876543209877, 4.82962962962963, 4.4641975308641975, 3.785185185185185], "corr": [[1.0, -0.1508189671823723, -0.03670650451650128, 0.3600988135558949, -0.04730419948454849, -0.11309748298045845, 0.3555886517348197, 0.6010301238723352, 0.020013778978110436, 0.81048298982295, 0.4246625876721974, 0.3608233533139674, 0.19056756485493276], [-0.1508189671823723, 1.0, 0.3906692827393407, -0.01103880724724483, 0.3845833780136831, 0.671747100812761, 0.02947977094270065, 0.12081759191765926, 0.09083607524263773, -0.053638603857954895, 0.21909641316804104, 0.22502166850151162, 0.2736744445791116], [-0.03670650451650128, 0.3906692827393407, 1.0, 0.13902752111324904, 0.9589668993506213, 0.2855540185555553, 0.03953790059844366, 0.08638774174869461, 0.0803641283517861, 0.017220084331020694, 0.3079761508522843, 0.2682905647371061, 0.3079026700640018], [0.3600988135558949, -0.01103880724724483, 0.13902752111324904, 1.0, 0.16311141134905974, -0.00031605663091768864, 0.788632344304093, 0.39181482943534796, 0.5022343051954669, 0.11492683998252251, 0.5585533703953763, 0.4960715051232829, 0.4706102136154386], [-0.04730419948454849, 0.3845833780136831, 0.9589668993506213, 0.16311141134905974, 1.0, 0.26614467550234416, 0.10955789843480995, 0.11630826314000853, 0.11353066132284291, -0.013493641762166718, 0.29752465876234424, 0.2814718967807393, 0.30446686920951077], [-0.11309748298045845, 0.671747100812761, 0.2855540185555553, -0.00031605663091768864, 0.26614467550234416, 1.0, -0.08229325595874307, 0.08007919689835999, 0.1900415457599812, -0.06454962432226657, 0.19555341768609052, 0.2729060713529077, 0.24839369147001028], [0.3555886517348197, 0.02947977094270065, 0.03953790059844366, 0.788632344304093, 0.10955789843480995, -0.08229325595874307, 1.0, 0.3811817020670222, 0.5885573110436184, 0.16189495970783127, 0.4399202611675235, 0.4538332063569838, 0.37161844744504485], [0.6010301238723352, 0.12081759191765926, 0.08638774174869461, 0.39181482943534796, 0.11630826314000853, 0.08007919689835999, 0.3811817020670222, 1.0, 0.18542447178008162, 0.5041027830032256, 0.39701910970386256, 0.4461322564458617, 0.3350016988696614], [0.020013778978110436, 0.09083607524263773, 0.0803641283517861, 0.5022343051954669, 0.11353066132284291, 0.1900415457599812, 0.5885573110436184, 0.18542447178008162, 1.0, 0.043149928781414666, 0.5043567303804077, 0.5173835199003213, 0.5049532081738316], [0.81048298982295, -0.053638603857954895, 0.017220084331020694, 0.11492683998252251, -0.013493641762166718, -0.06454962432226657, 0.16189495970783127, 0.5041027830032256, 0.043149928781414666, 1.0, 0.3837689104560289, 0.32381579434512703, 0.1633608985696127], [0.4246625876721974, 0.21909641316804104, 0.3079761508522843, 0.5585533703953763, 0.29752465876234424, 0.19555341768609052, 0.4399202611675235, 0.39701910970386256, 0.5043567303804077, 0.3837689104560289, 1.0, 0.8237918154769781, 0.8018375217777752], [0.3608233533139674, 0.22502166850151162, 0.2682905647371061, 0.4960715051232829, 0.2814718967807393, 0.2729060713529077, 0.4538332063569838, 0.4461322564458617, 0.5173835199003213, 0.32381579434512703, 0.8237918154769781, 1.0, 0.8441351302723302], [0.19056756485493276, 0.2736744445791116, 0.3079026700640018, 0.4706102136154386, 0.30446686920951077, 0.24839369147001028, 0.37161844744504485, 0.3350016988696614, 0.5049532081738316, 0.1633608985696127, 0.8018375217777752, 0.8441351302723302, 1.0]], "r_crit": 0.09746066407377929, "r_outcomes": [[0.4246625876721975, 0.3608233533139672, 0.19056756485493292], [0.21909641316804077, 0.2250216685015124, 0.2736744445791119], [0.30797615085228414, 0.26829056473710666, 0.3079026700640014], [0.5585533703953763, 0.4960715051232824, 0.4706102136154386], [0.29752465876234363, 0.28147189678073925, 0.30446686920951066], [0.19555341768609025, 0.27290607135290823, 0.24839369147001014], [0.4399202611675231, 0.45383320635698315, 0.3716184474450448], [0.39701910970386217, 0.44613225644586135, 0.33500169886966125], [0.5043567303804082, 0.5173835199003218, 0.5049532081738318], [0.3837689104560294, 0.32381579434512675, 0.16336089856961267]], "p_outcomes": [[3.661072144249256e-19, 6.737462130833513e-14, 0.00011400888611409623], [8.595755580055164e-06, 4.8019450885727355e-06, 2.170978009811032e-08], [2.399199239217991e-10, 4.1721568147035274e-08, 2.424042780040858e-10], [1.3365845429659199e-34, 1.5152478426602387e-26, 1.0332425433216635e-23], [1.0093418399832852e-09, 8.216809501707538e-09, 3.9115644128068526e-10], [7.445354384592545e-05, 2.385199859743843e-08, 4.1283647063391627e-07], [1.3453271894821238e-20, 5.699232507438903e-22, 1.0399230211437835e-14], [9.640304198684843e-17, 3.3387147243513332e-21, 4.454406539736597e-12], [1.606871395860792e-27, 4.157842670093706e-29, 1.3639278515696537e-27], [1.165713567342377e-15, 2.4342051723935197e-11, 0.0009682072925900858]], "n_strong_pairs": 8.0, "max_diff": 0.06842632143862737, "group_sizes": [85.0, 93.0, 60.0, 167.0], "familywise_error": 0.2649081093750002, "group_stats": [[85.0, 3.988235294117647, 0.5874504568180082, 4.0], [93.0, 4.354838709677419, 0.7752301382047306, 4.0], [60.0, 4.85, 1.2865009699812557, 5.0], [167.0, 5.514970059880239, 0.9930862653483282, 6.0]], "sd_ratio": 2.1899735629617747, "shapiro_p": [3.245111132689736e-14, 1.7616896812071658e-09, 0.00017488510184753006, 1.591492319839084e-11], "levene_F": 38.59652927376411, "levene_p": 6.22623025886254e-22, "SSB": 159.6033117193822, "SSW": 345.6411327250622, "SST": 505.2444444444444, "df_between": 3.0, "df_within": 401.0, "F_stat": 61.72194408198647, "p_value": 7.920364966704744e-33, "F_crit": 2.627158374419695, "eta_sq": 0.315893254194766, "welch_F": 80.89187232997844, "welch_p": 5.82961926153419e-33, "kw_H": 129.78593238202583, "kw_p": 6.014577647893677e-28, "tukey_p": [0.0436131890199809, 3.949278126746947e-07, 0.0, 0.007522891591577152, 0.0, 1.6163300856075757e-05]}

def check(name, value):
    """Compare your result with the full version (0.1% tolerance)."""
    exp = np.asarray(EXPECTED[name], dtype=float)
    try:
        got = np.asarray(value, dtype=float)
    except (TypeError, ValueError):
        print(f"[?] {name}: could not read your value as numbers")
        return
    if got.shape != exp.shape:
        print(f"[X] {name}: expected shape {exp.shape}, yours is {got.shape}")
    elif np.allclose(got, exp, rtol=1e-3, atol=1e-10):
        print(f"[OK] {name} matches the full version")
    else:
        print(f"[X] {name} does not match yet. Re-read the hint and try again.")


## 1. Load and Clean the Data

**Your Turn.** Load the codebook sheet `Questions` into `questions` (columns `code`, `question`) and the sheet `Survey Responses` into `raw`. Store `raw_shape = raw.shape` and print it with the column names.

*Hint:* `pd.read_excel(DATA_FILE, sheet_name=...)`. For the codebook use `header=None, usecols="A:B", skiprows=1, names=["code", "question"]`.

In [ ]:
DATA_FILE = "Customer_Satisfaction.xlsx"
# Your code here


In [ ]:
# Check
check("raw_shape", raw_shape)

**Your Turn.** Keep only `id` and `x12` to `x24` in a new DataFrame `df`. Then count: rows (`n_rows`), missing values (`n_missing`), duplicated ids (`n_duplicate_ids`) and answers outside 1 to 7 (`n_out_of_range`). Define `VARS`, `PERCEPTION`, `OUTCOMES` and the `SHORT` label dictionary exactly as described in the hint.

*Hint:* `VARS = [f"x{i}" for i in range(12, 25)]`, `PERCEPTION = VARS[:10]`, `OUTCOMES = ["x22", "x23", "x24"]`. `SHORT` maps each code to a short label (x12 Friendly employees, x13 Fun place to eat, x14 Large portions, x15 Fresh food, x16 Reasonable prices, x17 Attractive interior, x18 Excellent food taste, x19 Knowledgeable employees, x20 Proper food temperature, x21 Quick service, x22 Satisfaction, x23 Likely to return, x24 Likely to recommend). Missing: `.isna().sum().sum()`. Duplicates: `.duplicated().sum()`. Out of range: `((df[VARS] < 1) | (df[VARS] > 7)).sum().sum()`.

In [ ]:
# Your code here


In [ ]:
# Check
check("n_rows", n_rows)
check("n_missing", n_missing)
check("n_duplicate_ids", n_duplicate_ids)
check("n_out_of_range", n_out_of_range)

> **Defend It (say it out loud):** What did you check during cleaning, what did you find, and why did you drop the extra columns?

## 2. Descriptive Context

**Your Turn.** Build `desc`: one row per item with count, mean, standard deviation, median, min and max. Store the means column as `desc_means`.

*Hint:* `df[VARS].agg(["count", "mean", "std", "median", "min", "max"]).T`

In [ ]:
# Your code here


In [ ]:
# Check
check("desc_means", desc_means)

> **Defend It (say it out loud):** Which item is rated highest and lowest, and why does quick service have the largest spread?

# Part A. Correlation Analysis

**Task written in the dataset:** get the correlation of the perception indicators with each other, as
well as with customer satisfaction, likelihood to return, and likelihood to recommend.

**Association versus correlation.** *Association* is the general idea that two variables are related
in any way (including categorical variables and curved patterns). A *correlation coefficient* is a
specific number for the strength and direction of a **linear** relationship between two numeric
variables. Pearson's r runs from -1 (perfect negative line) through 0 (no linear relationship) to +1
(perfect positive line). Correlation says two things move together; it does not say either one causes
the other.

**Your Turn.** Compute the 13 x 13 Pearson correlation matrix of all items and store it as `corr`.

*Hint:* `df[VARS].corr(method="pearson")`

In [ ]:
# Your code here


In [ ]:
# Check
check("corr", corr)

**Your Turn.** Draw a heatmap of `corr` with the value printed in each cell (Figure 1).

*Hint:* `plt.imshow(corr, cmap=..., vmin=-1, vmax=1)` plus `ax.text(j, i, f"{r:.2f}")` in a double loop. Compare your chart with Figure 1 in the full version.

In [ ]:
# Your code here


> **Defend It (say it out loud):** How do you read one square of the heatmap, and what does the bottom-right block show?

**Your Turn.** Find `r_crit`, the smallest absolute correlation that is significant at alpha = 0.05 with this sample size.

*Hint:* Testing r uses t = r * sqrt(df / (1 - r^2)) with df = n - 2. Solve for r at the critical t: `t_crit = stats.t.ppf(1 - ALPHA/2, df_r)`, then `r_crit = t_crit / np.sqrt(t_crit**2 + df_r)`.

In [ ]:
# Your code here


In [ ]:
# Check
check("r_crit", r_crit)

> **Defend It (say it out loud):** Why are almost all the p-values tiny, and what should you look at instead?

**Your Turn.** For each perception item (x12 to x21) and each outcome (x22, x23, x24), get Pearson's r and its p-value. Store the r values as a 10 x 3 DataFrame `r_outcomes` (rows x12..x21, columns x22, x23, x24) and the p-values the same way as `p_outcomes`. Add a strength label.

*Hint:* Loop over both lists and call `stats.pearsonr(df[x], df[y])`; it returns `.statistic` and `.pvalue`. Collect rows, then `.pivot(index="indicator", columns="outcome", values="r")` and reorder with `.loc[PERCEPTION, OUTCOMES]`. Labels: below 0.10 negligible, 0.10 to 0.29 weak, 0.30 to 0.49 moderate, 0.50 to 0.69 strong, 0.70 and up very strong.

In [ ]:
# Your code here


In [ ]:
# Check
check("r_outcomes", r_outcomes)
check("p_outcomes", p_outcomes)

**Your Turn.** Draw a grouped horizontal bar chart of `r_outcomes` (Figure 2), one bar per outcome, items sorted by their correlation with satisfaction, and a dashed line at `r_crit`.

*Hint:* Use `ax.barh` three times with the bar positions shifted by the bar height. Compare with Figure 2 in the full version.

In [ ]:
# Your code here


> **Defend It (say it out loud):** Which perception is most related to satisfaction, which is least, and what do all 30 correlations have in common?

**Your Turn.** Among the ten perception items only, list every pair with |r| of 0.5 or more (each pair once). Store the count as `n_strong_pairs`.

*Hint:* Take `corr.loc[PERCEPTION, PERCEPTION]`, keep only the upper triangle with `np.triu(np.ones((10, 10), dtype=bool), k=1)` and `.where(mask)`, then `.stack()` and filter.

In [ ]:
# Your code here


In [ ]:
# Check
check("n_strong_pairs", n_strong_pairs)

> **Defend It (say it out loud):** Name the most strongly linked pair of perceptions and explain why it matters for Part B.

**Your Turn.** Robustness check: compute the Spearman (rank) correlation matrix as `spearman` and the largest absolute difference from Pearson as `max_diff`.

*Hint:* `df[VARS].corr(method="spearman")`, then `(spearman - corr).abs().max().max()`.

In [ ]:
# Your code here


In [ ]:
# Check
check("max_diff", max_diff)

> **Defend It (say it out loud):** Why check Spearman, and did it change anything?


**Part A Summary (write your own).** In three sentences: which perceptions matter most and least for
the outcomes, and why Part B focuses on fresh food.



## Task 1. Research Question, Hypotheses and Variables

**Your turn (write in this cell).** State the research question for comparing satisfaction (x22) across
freshness ratings (x15), then H0 and H1 in words and symbols, and a table of the two variables with their
role and measurement type.

*Hint:* H0 says all group means are equal; H1 says at least one differs (not "all differ").


## Task 2. Choosing the Test

**Your Turn.** Find the freshness levels present (`GROUP_LEVELS`), the group sizes (`group_sizes`), and the chance of at least one false positive if you ran a t-test for every pair of groups (`familywise_error`).

*Hint:* `sorted(df["x15"].unique())`, `df["x15"].value_counts().sort_index()`. With k groups there are k(k-1)/2 pairs; the familywise error is 1 - (1 - alpha)^pairs.

In [ ]:
# Your code here


In [ ]:
# Check
check("group_sizes", group_sizes)
check("familywise_error", familywise_error)

> **Defend It (say it out loud):** Why one-way ANOVA and not a t-test or a Chi-square test? Give one reason for each.

## Task 3. Checking the Assumptions

One-way ANOVA assumes (1) **independent observations**, (2) **roughly normal** satisfaction scores within
each group, and (3) **equal variances** across groups.

**Your Turn.** Make `groups`, a list with the satisfaction scores for each freshness level, and `group_stats`: n, mean, sd and median of x22 per freshness level (columns in that order). Compute `sd_ratio` = largest SD / smallest SD.

*Hint:* `df.groupby("x15")["x22"].agg(n="count", mean="mean", sd="std", median="median")`

In [ ]:
# Your code here


In [ ]:
# Check
check("group_stats", group_stats)
check("sd_ratio", sd_ratio)

**Your Turn.** Run a Shapiro-Wilk normality test on each group. Store the four p-values as `shapiro_p`.

*Hint:* `stats.shapiro(values)` returns (W, p). Loop over `groups`.

In [ ]:
# Your code here


In [ ]:
# Check
check("shapiro_p", shapiro_p)

**Your Turn.** Run Levene's test for equal variances across the four groups, mean-centered. Store `levene_F` and `levene_p`.

*Hint:* `stats.levene(*groups, center="mean")`. The default in SciPy is median-centered, which is a different (Brown-Forsythe) statistic; set `center="mean"` to match the Excel workbook.

In [ ]:
# Your code here


In [ ]:
# Check
check("levene_F", levene_F)
check("levene_p", levene_p)

**Your Turn.** Draw one histogram of satisfaction per freshness group, side by side (Figure 3).

*Hint:* `plt.subplots(1, 4, sharey=True)` and `ax.hist(values, bins=np.arange(0.5, 8.5, 1))`.

In [ ]:
# Your code here


> **Defend It (say it out loud):** Which assumptions are violated, how do you know, and what did you do about each one?

## Task 4. Running the Analysis

**Your Turn.** Compute the one-way ANOVA by hand: `SSB`, `SSW`, `SST`, `df_between`, `df_within`, `F_stat`, `p_value`, `F_crit` (critical F at alpha 0.05) and the effect size `eta_sq` = SSB / SST.

*Hint:* SSB = sum of n_g * (group mean - grand mean)^2. SSW = sum of squared deviations from each group's own mean. F = (SSB/df_between) / (SSW/df_within). p-value: `stats.f.sf(F, df1, df2)`. Critical F: `stats.f.ppf(1 - ALPHA, df1, df2)`.

In [ ]:
# Your code here


In [ ]:
# Check
check("SSB", SSB)
check("SSW", SSW)
check("SST", SST)
check("df_between", df_between)
check("df_within", df_within)
check("F_stat", F_stat)
check("p_value", p_value)
check("F_crit", F_crit)
check("eta_sq", eta_sq)

**Your Turn.** Verify your hand calculation with `stats.f_oneway(*groups)` and confirm SSB + SSW = SST.

*Hint:* The F and p from `f_oneway` should equal yours.

In [ ]:
# Your code here


> **Defend It (say it out loud):** In plain words, what does F compare, and what does eta squared tell you that the p-value does not?

**Your Turn.** Run the two remedies: Welch's ANOVA (`welch_F`, `welch_p`) and Kruskal-Wallis (`kw_H`, `kw_p`).

*Hint:* `stats.f_oneway(*groups, equal_var=False)` gives Welch. `stats.kruskal(*groups)` gives Kruskal-Wallis.

In [ ]:
# Your code here


In [ ]:
# Check
check("welch_F", welch_F)
check("welch_p", welch_p)
check("kw_H", kw_H)
check("kw_p", kw_p)

> **Defend It (say it out loud):** Why run Welch and Kruskal-Wallis, and what would you have concluded if they disagreed with the ANOVA?

**Your Turn.** Post-hoc test: run Tukey's HSD to see which pairs of freshness levels differ. Store the six pairwise p-values as `tukey_p` in the order 5 vs 4, 6 vs 4, 7 vs 4, 6 vs 5, 7 vs 5, 7 vs 6.

*Hint:* `stats.tukey_hsd(*groups)`; the p-value of groups i and j is `.pvalue[i, j]`. Loop i < j.

In [ ]:
# Your code here


In [ ]:
# Check
check("tukey_p", tukey_p)

> **Defend It (say it out loud):** What does the post-hoc test add to the ANOVA, and which comparison is the weakest?

**Your Turn.** Plot the mean satisfaction per freshness level with 95% confidence-interval error bars, and the individual answers as faint jittered dots behind them (Figure 4).

*Hint:* CI half-width = t(0.975, n - 1) * sd / sqrt(n). Use `ax.errorbar(levels, means, yerr=...)`. Compare with Figure 4 in the full version.

In [ ]:
# Your code here


> **Defend It (say it out loud):** What do the blue dots, the bars and the grey dots each show?

**Your Turn.** Print the decision (reject or fail to reject H0) with F, degrees of freedom, p-value and eta squared.

*Hint:* `"Reject H0" if p_value < ALPHA else "Fail to reject H0"`

In [ ]:
# Your code here



**Plain-Language Conclusion (write your own).** One or two sentences a restaurant manager would
understand, including F, the degrees of freedom, the p-value and what it means at alpha = 0.05.



## Task 5. Diagnostic Analytics Report

**Your turn (write in this cell).** Cover, in short paragraphs or bullets:

1. **What happened** (Part A in two sentences).
2. **Why**: the ANOVA result with F, df and p, and that Welch and Kruskal-Wallis agree.
3. **Statistical versus practical significance**: use eta squared and the 7-vs-4 mean difference, and
   explain why the p-value alone is not enough with n = 405.
4. **Cautions**: survey not experiment; freshness overlaps with taste; nobody rated freshness below 4.
5. **One reasonable next step.**
6. A **60-second oral explanation** you can say without notes.

*Hint:* every number you quote should be one your own cells printed. Compare with the full version
only after writing yours.



## Question and Answer

Answer each in three to five sentences, in your own words, using this analysis as the example.

**1. Why does rejecting the null hypothesis not automatically prove that one variable causes another?**

*Your answer:*

**2. How should a researcher respond when the assumptions of the selected test are not satisfied?**

*Your answer:*

**3. Why should statistical significance be discussed together with practical significance?**

*Your answer:*

*Hint:* for 1, think survey versus experiment and the freshness and taste overlap. For 2, think "report,
judge how serious, remedy, compare". For 3, think about what n = 405 does to p-values.



## Appendix: AI Tools and Prompts

Use the appendix in the full version, and add anything you did differently while practising.
